# 05b — Per-Bucket Covered Call Returns

Computes the **realized covered call return** for each (ticker, month, moneyness bucket).

This extends notebook 05 (label construction) which only kept the *best* bucket's return.
Here we keep returns for **all three** moneyness buckets (ATM, OTM5, OTM10) so the
backtesting strategy can evaluate the return of any specific bucket choice — not just the oracle.

**Covered call payoff per contract:**
```
payoff = premium + min(strike, stock_at_expiry) - stock_at_entry
return = payoff / stock_at_entry
```

**Important:** Uses **unadjusted `close`** (not `adjusted_close`) for stock prices, because
option strikes are in nominal (pre-split) terms. Using split-adjusted prices against nominal
strikes produces wildly inflated returns.

**Output:** `data/processed/bucket_returns.parquet` — one row per (ticker, month), with columns
`return_ATM`, `return_OTM5`, `return_OTM10`.

In [ ]:
import pandas as pd
import numpy as np

## 1. Load data

In [ ]:
opts = pd.read_parquet('../data/processed/options_clean.parquet')
daily = pd.read_parquet('../data/processed/daily_clean.parquet')
daily['date'] = pd.to_datetime(daily['date'])

calls = opts[opts['call_put'] == 'CALL'].copy()
calls['trade_date'] = pd.to_datetime(calls['trade_date'])
calls['expiration'] = pd.to_datetime(calls['expiration'])
calls['dte'] = (calls['expiration'] - calls['trade_date']).dt.days
calls['year_month'] = calls['trade_date'].dt.to_period('M')

print(f'Total call contracts: {len(calls)}')

## 2. Assign moneyness buckets by delta

Same ranges as notebook 05:
- **OTM10**: delta 0.15–0.30
- **OTM5**: delta 0.30–0.45
- **ATM**: delta 0.45–0.60

In [ ]:
calls['moneyness'] = pd.cut(
    calls['delta'],
    bins=[0.15, 0.30, 0.45, 0.60],
    labels=['OTM10', 'OTM5', 'ATM'],
)
calls = calls.dropna(subset=['moneyness'])

# Assign maturity bucket by DTE (same threshold as model.py iv_rank rule)
calls['maturity'] = np.where(calls['dte'] <= 45, 'SHORT', 'LONG')

print(f'Calls with moneyness assigned: {len(calls)}')
print(calls['moneyness'].value_counts())
print(f'\nMaturity split:')
print(calls['maturity'].value_counts())

## 3. Merge stock prices at entry and expiry

In [ ]:
# Stock price at entry — use UNADJUSTED close (strikes are nominal)
calls = calls.merge(
    daily[['symbol', 'date', 'close']].rename(
        columns={'date': 'trade_date', 'close': 'stock_entry'}
    ),
    on=['symbol', 'trade_date'],
    how='left',
)
print(f'Entry price nulls: {calls["stock_entry"].isna().sum()}')

# Stock price at expiry (last unadjusted close on or before expiration)
calls = calls.sort_values('expiration')
daily_sorted = daily[['symbol', 'date', 'close']].sort_values('date')
daily_sorted = daily_sorted.rename(columns={'close': 'stock_expiry'})

calls = pd.merge_asof(
    calls,
    daily_sorted,
    left_on='expiration',
    right_on='date',
    by='symbol',
    direction='backward',
)
print(f'Expiry price nulls: {calls["stock_expiry"].isna().sum()}')

## 4. Compute covered call return per contract

```
payoff = premium + min(strike, stock_at_expiry) - stock_at_entry
return = payoff / stock_at_entry
```

Filter outliers to [-50%, +50%] (same as notebook 05).

In [ ]:
calls['cc_payoff'] = calls['mark'] + np.minimum(calls['strike'], calls['stock_expiry']) - calls['stock_entry']
calls['cc_return'] = calls['cc_payoff'] / calls['stock_entry']

before = len(calls)
calls = calls[(calls['cc_return'] >= -0.50) & (calls['cc_return'] <= 0.50)]
print(f'After outlier filter: {len(calls)} (dropped {before - len(calls)})')
print(f'Return stats:\n{calls["cc_return"].describe()}')

## 5. Aggregate returns

Two aggregations:
1. **Moneyness-only** — average return per (ticker, month, moneyness). Used by Argmax, Risk-Adjusted, Baseline.
2. **Moneyness × Maturity** — average return per (ticker, month, moneyness, maturity). Used by Conservative, Balanced, Aggressive presets.

In [ ]:
# Moneyness-only aggregation
bucket_returns = (
    calls.groupby(['symbol', 'year_month', 'moneyness'], observed=True)
    .agg(
        bucket_return=('cc_return', 'mean'),
        n_contracts=('cc_return', 'count'),
    )
    .reset_index()
)
print(f'Moneyness-only rows: {len(bucket_returns)}')

# Moneyness × Maturity aggregation
bucket_returns_mat = (
    calls.groupby(['symbol', 'year_month', 'moneyness', 'maturity'], observed=True)
    .agg(
        bucket_return=('cc_return', 'mean'),
        n_contracts=('cc_return', 'count'),
    )
    .reset_index()
)
bucket_returns_mat['full_bucket'] = bucket_returns_mat['moneyness'].astype(str) + '_' + bucket_returns_mat['maturity']
print(f'Moneyness × Maturity rows: {len(bucket_returns_mat)}')
print(f'\nFull bucket distribution:')
print(bucket_returns_mat['full_bucket'].value_counts().sort_index())

## 6. Pivot to wide format and save

Output has 9 return columns:
- `return_ATM`, `return_OTM5`, `return_OTM10` (moneyness-only)
- `return_ATM_SHORT`, `return_ATM_LONG`, `return_OTM5_SHORT`, `return_OTM5_LONG`, `return_OTM10_SHORT`, `return_OTM10_LONG` (moneyness × maturity)

In [ ]:
# Pivot moneyness-only
pivot_mono = bucket_returns.pivot_table(
    index=['symbol', 'year_month'],
    columns='moneyness',
    values='bucket_return',
    observed=True,
).reset_index()
pivot_mono.columns.name = None
pivot_mono = pivot_mono.rename(columns={
    'ATM': 'return_ATM', 'OTM5': 'return_OTM5', 'OTM10': 'return_OTM10',
})

# Pivot moneyness × maturity
pivot_mat = bucket_returns_mat.pivot_table(
    index=['symbol', 'year_month'],
    columns='full_bucket',
    values='bucket_return',
    observed=True,
).reset_index()
pivot_mat.columns.name = None
pivot_mat = pivot_mat.rename(columns={
    'ATM_SHORT': 'return_ATM_SHORT', 'ATM_LONG': 'return_ATM_LONG',
    'OTM5_SHORT': 'return_OTM5_SHORT', 'OTM5_LONG': 'return_OTM5_LONG',
    'OTM10_SHORT': 'return_OTM10_SHORT', 'OTM10_LONG': 'return_OTM10_LONG',
})

# Merge both pivots
pivot = pivot_mono.merge(pivot_mat.drop(columns=['symbol', 'year_month'], errors='ignore'),
                         left_index=True, right_index=True)

# If merge lost alignment, do it properly on keys
if 'return_ATM_SHORT' not in pivot.columns:
    pivot = pivot_mono.merge(pivot_mat, on=['symbol', 'year_month'], how='outer')

print(f'Shape: {pivot.shape}')
print(f'Columns: {pivot.columns.tolist()}')
print(f'\nNull counts:')
ret_cols = [c for c in pivot.columns if c.startswith('return_')]
print(pivot[ret_cols].isna().sum())
print()
pivot.head(10)

In [ ]:
pivot.to_parquet('../data/bucket_returns.parquet', index=False)
bucket_returns_mat.to_parquet('../data/bucket_returns_detail.parquet', index=False)
print('Saved: src/data/bucket_returns.parquet (9 return columns)')
print('Saved: src/data/bucket_returns_detail.parquet (moneyness × maturity detail)')

## 7. Quick sanity check

Compare with `best_return` from modeling_data — our per-bucket max should match.

In [ ]:
modeling = pd.read_parquet('../data/processed/modeling_data.parquet')
modeling['year_month'] = pd.to_datetime(modeling['decision_date']).dt.to_period('M')

# Our best = max across 3 moneyness-only buckets
pivot['our_best'] = pivot[['return_ATM', 'return_OTM5', 'return_OTM10']].max(axis=1)

check = pivot.merge(modeling[['symbol', 'year_month', 'best_return']], on=['symbol', 'year_month'], how='inner')
print(f'Matched rows: {len(check)}')
print(f'Correlation between our best and modeling best_return: {check["our_best"].corr(check["best_return"]):.4f}')
print(f'Mean difference: {(check["our_best"] - check["best_return"]).mean():.6f}')

# Sanity: maturity columns should sum to roughly the moneyness-only column
print(f'\nMaturity sanity check (ATM):')
has_both = pivot[pivot['return_ATM_SHORT'].notna() & pivot['return_ATM_LONG'].notna()]
print(f'  Rows with both SHORT and LONG: {len(has_both)}')